#instalando e Iniciando o Pyspark

In [1]:
# Install Java
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Download and extract Spark
!wget -q http://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz

# Install PySpark
!pip install pyspark==3.5.0

# Set environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Initialize SparkSession
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425346 sha256=19164db37ae1d23d0b7206176f1af58373af125d8f5ded673a509b246a5a206f
  Stored in directory: /root/.cache/pip/wheels/84/40/20/65eefe766118e0a8f8e385cc3ed6e9eb7241c7e51cfc04c51a
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.1
    Uninstalling pyspark-3.5.1:
      Successfully uninstalled pyspark-3.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires pyspark[connect]~=3.5.1, but you have pyspark 3.5.0 which is incompatible.


In [2]:
from pyspark.sql import functions as f
from pyspark.sql.types import *

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
path = "/content/drive/MyDrive/Curso_Spark/empresas"
empresas = spark.read.csv(path, sep=";", inferSchema=True)

In [7]:
path = "/content/drive/MyDrive/Curso_Spark/socios"
socios = spark.read.csv(path, sep=";", inferSchema=True)

In [8]:
path = "/content/drive/MyDrive/Curso_Spark/estabelecimentos"
estabelecimentos = spark.read.csv(path, sep=";", inferSchema=True)

In [9]:
estabelecimentos.limit(5).toPandas() # similar ao head

,_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,...,_c20,_c21,_c22,_c23,_c24,_c25,_c26,_c27,_c28,_c29
0,1879,1,96,1,PIRAMIDE M. C.,8,20011029,1,None,NaN,...,7107,None,None,None,None,NaN,None,None,None,NaN
1,2818,1,43,1,None,8,20081231,71,None,NaN,...,7107,None,None,None,None,NaN,None,None,None,NaN
2,3110,1,7,1,None,8,19971231,1,None,NaN,...,7107,None,None,None,None,NaN,None,None,None,NaN
3,3733,1,80,1,None,8,20081231,71,None,NaN,...,7107,None,None,None,None,NaN,None,None,None,NaN
4,4628,3,27,2,EMBROIDERY & GIFT,8,19980429,1,None,NaN,...,7075,None,None,None,None,NaN,None,None,None,NaN


#Renomeando

In [10]:
empresasColNames = ['cnpj_basico', 'razao_social_nome_empresarial', 'natureza_juridica', 'qualificacao_do_responsavel', 'capital_social_da_empresa', 'porte_da_empresa', 'ente_federativo_responsavel']

In [11]:
for item in enumerate(empresasColNames):
  empresas = empresas.withColumnRenamed(f"_c{item[0]}", item[1])

In [12]:
estabsColNames = ['cnpj_basico', 'cnpj_ordem', 'cnpj_dv', 'identificador_matriz_filial', 'nome_fantasia', 'situacao_cadastral', 'data_situacao_cadastral', 'motivo_situacao_cadastral', 'nome_da_cidade_no_exterior', 'pais', 'data_de_inicio_atividade', 'cnae_fiscal_principal', 'cnae_fiscal_secundaria', 'tipo_de_logradouro', 'logradouro', 'numero', 'complemento', 'bairro', 'cep', 'uf', 'municipio', 'ddd_1', 'telefone_1', 'ddd_2', 'telefone_2', 'ddd_do_fax', 'fax', 'correio_eletronico', 'situacao_especial', 'data_da_situacao_especial']

In [13]:
for item in enumerate(estabsColNames):
  estabelecimentos = estabelecimentos.withColumnRenamed(f"_c{item[0]}", item[1])

In [14]:
sociosColNames = ['cnpj_basico', 'identificador_de_socio', 'nome_do_socio_ou_razao_social', 'cnpj_ou_cpf_do_socio', 'qualificacao_do_socio', 'data_de_entrada_sociedade', 'pais', 'representante_legal', 'nome_do_representante', 'qualificacao_do_representante_legal', 'faixa_etaria']

In [15]:
for item in enumerate(sociosColNames):
  socios = socios.withColumnRenamed(f"_c{item[0]}", item[1])

#Convertendo String - Date

In [16]:
df = spark.createDataFrame([(20200924,), (20201022,), (20210215,)], ['data'])
df.toPandas()

,data
0,20200924
1,20201022
2,20210215


In [17]:
df = df.withColumn('data', f.to_date(df.data.cast(StringType()), 'yyyyMMdd'))

In [18]:
df.printSchema()

root
 |-- data: date (nullable = true)



In [19]:
df.toPandas()

,data
0,2020-09-24
1,2020-10-22
2,2021-02-15


In [20]:
estabelecimentos.printSchema()

root
 |-- cnpj_basico: integer (nullable = true)
 |-- cnpj_ordem: integer (nullable = true)
 |-- cnpj_dv: integer (nullable = true)
 |-- identificador_matriz_filial: integer (nullable = true)
 |-- nome_fantasia: string (nullable = true)
 |-- situacao_cadastral: integer (nullable = true)
 |-- data_situacao_cadastral: integer (nullable = true)
 |-- motivo_situacao_cadastral: integer (nullable = true)
 |-- nome_da_cidade_no_exterior: string (nullable = true)
 |-- pais: integer (nullable = true)
 |-- data_de_inicio_atividade: integer (nullable = true)
 |-- cnae_fiscal_principal: integer (nullable = true)
 |-- cnae_fiscal_secundaria: string (nullable = true)
 |-- tipo_de_logradouro: string (nullable = true)
 |-- logradouro: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- complemento: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- cep: integer (nullable = true)
 |-- uf: string (nullable = true)
 |-- municipio: integer (nullable = true)
 |-- ddd_1: str

Usando

withColumn('variavel', f.to_date(data.variavel.cast(StringType()), 'yyyyMMdd'))


Para alterar a coluna para o formato Date

In [21]:
estabelecimentos = estabelecimentos.withColumn(
    'data_situacao_cadastral', f.to_date(estabelecimentos.data_situacao_cadastral.cast(StringType()), 'yyyyMMdd')).withColumn(
        'data_de_inicio_atividade', f.to_date(estabelecimentos.data_de_inicio_atividade.cast(StringType()), 'yyyyMMdd')).withColumn(
            'data_da_situacao_especial', f.to_date(estabelecimentos.data_da_situacao_especial.cast(StringType()), 'yyyyMMdd'))


In [22]:
estabelecimentos.limit(5).toPandas()

,cnpj_basico,cnpj_ordem,cnpj_dv,identificador_matriz_filial,nome_fantasia,situacao_cadastral,data_situacao_cadastral,motivo_situacao_cadastral,nome_da_cidade_no_exterior,pais,...,municipio,ddd_1,telefone_1,ddd_2,telefone_2,ddd_do_fax,fax,correio_eletronico,situacao_especial,data_da_situacao_especial
0,1879,1,96,1,PIRAMIDE M. C.,8,2001-10-29,1,None,NaN,...,7107,None,None,None,None,NaN,None,None,None,None
1,2818,1,43,1,None,8,2008-12-31,71,None,NaN,...,7107,None,None,None,None,NaN,None,None,None,None
2,3110,1,7,1,None,8,1997-12-31,1,None,NaN,...,7107,None,None,None,None,NaN,None,None,None,None
3,3733,1,80,1,None,8,2008-12-31,71,None,NaN,...,7107,None,None,None,None,NaN,None,None,None,None
4,4628,3,27,2,EMBROIDERY & GIFT,8,1998-04-29,1,None,NaN,...,7075,None,None,None,None,NaN,None,None,None,None


In [23]:
socios.printSchema()

root
 |-- cnpj_basico: integer (nullable = true)
 |-- identificador_de_socio: integer (nullable = true)
 |-- nome_do_socio_ou_razao_social: string (nullable = true)
 |-- cnpj_ou_cpf_do_socio: string (nullable = true)
 |-- qualificacao_do_socio: integer (nullable = true)
 |-- data_de_entrada_sociedade: integer (nullable = true)
 |-- pais: integer (nullable = true)
 |-- representante_legal: string (nullable = true)
 |-- nome_do_representante: string (nullable = true)
 |-- qualificacao_do_representante_legal: integer (nullable = true)
 |-- faixa_etaria: integer (nullable = true)



In [24]:
socios = socios.withColumn(
    'data_de_entrada_sociedade', f.to_date(socios.data_de_entrada_sociedade.cast(StringType()), 'yyyyMMdd'))

In [25]:
socios.limit(5).toPandas()

,cnpj_basico,identificador_de_socio,nome_do_socio_ou_razao_social,cnpj_ou_cpf_do_socio,qualificacao_do_socio,data_de_entrada_sociedade,pais,representante_legal,nome_do_representante,qualificacao_do_representante_legal,faixa_etaria
0,411,2,LILIANA PATRICIA GUASTAVINO,***678188**,22,1994-07-25,NaN,***000000**,None,0,7
1,411,2,CRISTINA HUNDERTMARK,***637848**,28,1994-07-25,NaN,***000000**,None,0,7
2,5813,2,CELSO EDUARDO DE CASTRO STEPHAN,***786068**,49,1994-05-16,NaN,***000000**,None,0,8
3,5813,2,EDUARDO BERRINGER STEPHAN,***442348**,49,1994-05-16,NaN,***000000**,None,0,5
4,14798,2,HANNE MAHFOUD FADEL,***760388**,49,1994-06-09,NaN,***000000**,None,0,8


# Realizando Consultas

.select('variaveis').show()

In [26]:
empresas\
.select("cnpj_basico", "razao_social_nome_empresarial").show()

+-----------+-----------------------------+
|cnpj_basico|razao_social_nome_empresarial|
+-----------+-----------------------------+
|        306|         FRANCAMAR REFRIGE...|
|       1355|         BRASILEIRO & OLIV...|
|       4820|         REGISTRO DE IMOVE...|
|       5347|         ROSELY APARECIDA ...|
|       6846|         BADU E FILHOS TEC...|
|       8416|           ELETRICA RUBI LTDA|
|       8992|         SHIROMA VEICULOS ...|
|       9091|         CONTATOS BAR E LA...|
|       9614|         ANTONIA APARECIDA...|
|       9896|         DORACY CORAT DA C...|
|      12112|         LANCHONETE RIO VE...|
|      12605|         VALMAR JACAREI CO...|
|      13407|         ROSANA CRISTINA D...|
|      13408|         CELIO RODRIGUES D...|
|      13721|         MAQFRAN COMERCIO ...|
|      21181|         MOURA & SILVA MER...|
|      21858|         PAMARATI COMERCIO...|
|      22277|         INACIO RODRIGUES ...|
|      24205|         SUELY LEME MARI A...|
|      26805|         SUELI BATI

In [27]:
empresas\
.select("cnpj_basico", "razao_social_nome_empresarial").show(5,False) #False para não truncar e mostrar a linha inteira

+-----------+--------------------------------------------------------------------------------------------+
|cnpj_basico|razao_social_nome_empresarial                                                               |
+-----------+--------------------------------------------------------------------------------------------+
|306        |FRANCAMAR REFRIGERACAO TECNICA S/C LTDA                                                     |
|1355       |BRASILEIRO & OLIVEIRA LTDA                                                                  |
|4820       |REGISTRO DE IMOVEIS, TABELIONATO 1 DE NOTAS E TABELIONATO E REGISTRO DE CONSTRATOS MARITIMOS|
|5347       |ROSELY APARECIDA MONTEIRO CALTABIANO FREITAS                                                |
|6846       |BADU E FILHOS TECIDOS LTDA                                                                  |
+-----------+--------------------------------------------------------------------------------------------+
only showing top 5 rows



In [28]:
empresas.select('*').show(10,False)

+-----------+--------------------------------------------------------------------------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+
|cnpj_basico|razao_social_nome_empresarial                                                               |natureza_juridica|qualificacao_do_responsavel|capital_social_da_empresa|porte_da_empresa|ente_federativo_responsavel|
+-----------+--------------------------------------------------------------------------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+
|306        |FRANCAMAR REFRIGERACAO TECNICA S/C LTDA                                                     |2240             |49                         |0,00                     |1               |NULL                       |
|1355       |BRASILEIRO & OLIVEIRA LTDA                                                                 

In [29]:
socios.printSchema()

root
 |-- cnpj_basico: integer (nullable = true)
 |-- identificador_de_socio: integer (nullable = true)
 |-- nome_do_socio_ou_razao_social: string (nullable = true)
 |-- cnpj_ou_cpf_do_socio: string (nullable = true)
 |-- qualificacao_do_socio: integer (nullable = true)
 |-- data_de_entrada_sociedade: date (nullable = true)
 |-- pais: integer (nullable = true)
 |-- representante_legal: string (nullable = true)
 |-- nome_do_representante: string (nullable = true)
 |-- qualificacao_do_representante_legal: integer (nullable = true)
 |-- faixa_etaria: integer (nullable = true)



Usando

f.year(variavel).alias(novo nome))

para pegar o ano da coluna e mudando o nome da coluna

In [30]:
socios\
.select('nome_do_socio_ou_razao_social', 'faixa_etaria', f.year('data_de_entrada_sociedade').alias ('ano_de_entrada'))\
.show(5, False)

+-------------------------------+------------+--------------+
|nome_do_socio_ou_razao_social  |faixa_etaria|ano_de_entrada|
+-------------------------------+------------+--------------+
|LILIANA PATRICIA GUASTAVINO    |7           |1994          |
|CRISTINA HUNDERTMARK           |7           |1994          |
|CELSO EDUARDO DE CASTRO STEPHAN|8           |1994          |
|EDUARDO BERRINGER STEPHAN      |5           |1994          |
|HANNE MAHFOUD FADEL            |8           |1994          |
+-------------------------------+------------+--------------+
only showing top 5 rows



# Identificando valores nulos

Valores Null/ None são diferente de Nan (Not a Number)

In [31]:
df = spark.createDataFrame([(1,), (2,), (3,), (None,)], ['data'])
df.toPandas()

,data
0,1.0
1,2.0
2,3.0
3,NaN


In [32]:
df.show()

+----+
|data|
+----+
|   1|
|   2|
|   3|
|NULL|
+----+



.toPandas() sempre mostra NaN

In [33]:
df = spark.createDataFrame([(1.,), (2.,), (3.,), (float('nan'),)], ['data'])
df.toPandas()

,data
0,1.0
1,2.0
2,3.0
3,NaN


NaN só aparecem em dados flutuantes/numericos

In [34]:
df.show()

+----+
|data|
+----+
| 1.0|
| 2.0|
| 3.0|
| NaN|
+----+



In [35]:
socios.limit(5).toPandas()

,cnpj_basico,identificador_de_socio,nome_do_socio_ou_razao_social,cnpj_ou_cpf_do_socio,qualificacao_do_socio,data_de_entrada_sociedade,pais,representante_legal,nome_do_representante,qualificacao_do_representante_legal,faixa_etaria
0,411,2,LILIANA PATRICIA GUASTAVINO,***678188**,22,1994-07-25,NaN,***000000**,None,0,7
1,411,2,CRISTINA HUNDERTMARK,***637848**,28,1994-07-25,NaN,***000000**,None,0,7
2,5813,2,CELSO EDUARDO DE CASTRO STEPHAN,***786068**,49,1994-05-16,NaN,***000000**,None,0,8
3,5813,2,EDUARDO BERRINGER STEPHAN,***442348**,49,1994-05-16,NaN,***000000**,None,0,5
4,14798,2,HANNE MAHFOUD FADEL,***760388**,49,1994-06-09,NaN,***000000**,None,0,8


In [36]:
socios.limit(5).show()

+-----------+----------------------+-----------------------------+--------------------+---------------------+-------------------------+----+-------------------+---------------------+-----------------------------------+------------+
|cnpj_basico|identificador_de_socio|nome_do_socio_ou_razao_social|cnpj_ou_cpf_do_socio|qualificacao_do_socio|data_de_entrada_sociedade|pais|representante_legal|nome_do_representante|qualificacao_do_representante_legal|faixa_etaria|
+-----------+----------------------+-----------------------------+--------------------+---------------------+-------------------------+----+-------------------+---------------------+-----------------------------------+------------+
|        411|                     2|         LILIANA PATRICIA ...|         ***678188**|                   22|               1994-07-25|NULL|        ***000000**|                 NULL|                                  0|           7|
|        411|                     2|         CRISTINA HUNDERTMARK|      

Função para contar a quantidaded de valores NULL

In [37]:
socios.select([f.count(f.when(f.isnull(c), 1)).alias(c) for c in socios.columns]).show()

+-----------+----------------------+-----------------------------+--------------------+---------------------+-------------------------+-------+-------------------+---------------------+-----------------------------------+------------+
|cnpj_basico|identificador_de_socio|nome_do_socio_ou_razao_social|cnpj_ou_cpf_do_socio|qualificacao_do_socio|data_de_entrada_sociedade|   pais|representante_legal|nome_do_representante|qualificacao_do_representante_legal|faixa_etaria|
+-----------+----------------------+-----------------------------+--------------------+---------------------+-------------------------+-------+-------------------+---------------------+-----------------------------------+------------+
|          0|                     0|                          208|                1234|                    0|                        1|2038255|                  0|              1995432|                                  0|           0|
+-----------+----------------------+------------------------

Função útil para verificar e depois confirmar o tipo de dado

Função para substituir todos NULL para 0
Usar com cuidado, pois colunas em formato string não serão alteradas

In [38]:
socios.na.fill(0).limit(5).toPandas()

,cnpj_basico,identificador_de_socio,nome_do_socio_ou_razao_social,cnpj_ou_cpf_do_socio,qualificacao_do_socio,data_de_entrada_sociedade,pais,representante_legal,nome_do_representante,qualificacao_do_representante_legal,faixa_etaria
0,411,2,LILIANA PATRICIA GUASTAVINO,***678188**,22,1994-07-25,0,***000000**,None,0,7
1,411,2,CRISTINA HUNDERTMARK,***637848**,28,1994-07-25,0,***000000**,None,0,7
2,5813,2,CELSO EDUARDO DE CASTRO STEPHAN,***786068**,49,1994-05-16,0,***000000**,None,0,8
3,5813,2,EDUARDO BERRINGER STEPHAN,***442348**,49,1994-05-16,0,***000000**,None,0,5
4,14798,2,HANNE MAHFOUD FADEL,***760388**,49,1994-06-09,0,***000000**,None,0,8


Para substituir todos os NULL, até os em formato string, rodar novamente com -

In [39]:
socios.na.fill('-').limit(5).toPandas()

,cnpj_basico,identificador_de_socio,nome_do_socio_ou_razao_social,cnpj_ou_cpf_do_socio,qualificacao_do_socio,data_de_entrada_sociedade,pais,representante_legal,nome_do_representante,qualificacao_do_representante_legal,faixa_etaria
0,411,2,LILIANA PATRICIA GUASTAVINO,***678188**,22,1994-07-25,NaN,***000000**,-,0,7
1,411,2,CRISTINA HUNDERTMARK,***637848**,28,1994-07-25,NaN,***000000**,-,0,7
2,5813,2,CELSO EDUARDO DE CASTRO STEPHAN,***786068**,49,1994-05-16,NaN,***000000**,-,0,8
3,5813,2,EDUARDO BERRINGER STEPHAN,***442348**,49,1994-05-16,NaN,***000000**,-,0,5
4,14798,2,HANNE MAHFOUD FADEL,***760388**,49,1994-06-09,NaN,***000000**,-,0,8


#Ordenando os dados

.orderBy('variavel', ascending=False)



 para ordernar

In [40]:
socios\
.select('nome_do_socio_ou_razao_social', 'faixa_etaria', f.year('data_de_entrada_sociedade').alias ('ano_de_entrada'))\
.orderBy('ano_de_entrada', ascending=False)\
.show(5, False)

+----------------------------------------+------------+--------------+
|nome_do_socio_ou_razao_social           |faixa_etaria|ano_de_entrada|
+----------------------------------------+------------+--------------+
|JOSE HUMBERTO PAIVA                     |6           |2021          |
|KASSIANO RODRIGO KICHILESKI             |4           |2021          |
|BENILDES BARBOSA RODRIGUES              |8           |2021          |
|LEONARDO MENNA BARRETO LARANJA GONCALVES|5           |2021          |
|MARCELO MOCELIN                         |5           |2021          |
+----------------------------------------+------------+--------------+
only showing top 5 rows



In [41]:
socios\
.select('nome_do_socio_ou_razao_social', 'faixa_etaria', f.year('data_de_entrada_sociedade').alias ('ano_de_entrada'))\
.orderBy('nome_do_socio_ou_razao_social')\
.show(5, False)

+-----------------------------+------------+--------------+
|nome_do_socio_ou_razao_social|faixa_etaria|ano_de_entrada|
+-----------------------------+------------+--------------+
|NULL                         |0           |2005          |
|NULL                         |0           |2005          |
|NULL                         |0           |2005          |
|NULL                         |0           |2005          |
|NULL                         |0           |2005          |
+-----------------------------+------------+--------------+
only showing top 5 rows



Se passar uma lista em .orderBy podemos ordernar mais de uma coluna


Também permite definir o tipo de ordenação das duas colunas se passar em lista False ou True em ascending

In [42]:
socios\
.select('nome_do_socio_ou_razao_social', 'faixa_etaria', f.year('data_de_entrada_sociedade').alias ('ano_de_entrada'))\
.orderBy(['ano_de_entrada', 'faixa_etaria'], ascending=[False, True])\
.show(5, False)

+-------------------------------------------------+------------+--------------+
|nome_do_socio_ou_razao_social                    |faixa_etaria|ano_de_entrada|
+-------------------------------------------------+------------+--------------+
|KALLAS ARKHES INCOPORACOES E CONSTRUCOES LTDA    |0           |2021          |
|TEXTIL RIO DOS CEDROS LTDA                       |0           |2021          |
|WEILLER CONSTRUCAO CIVIL LTDA                    |0           |2021          |
|FIO E FERRO MATERIAIS SERVICOS E CONSTRUCOES LTDA|0           |2021          |
|MARITIMA MERCOFLET SOCIEDAD ANONIMA              |0           |2021          |
+-------------------------------------------------+------------+--------------+
only showing top 5 rows



Lição


Df com nome e data de nascimento dos alunos

In [43]:
data = [
    ('CARMINA RABELO', 4, 2010),
    ('HERONDINA PEREIRA', 6, 2009),
    ('IRANI DOS SANTOS', 12, 2010),
    ('JOAO BOSCO DA FONSECA', 3, 2009),
    ('CARLITO SOUZA', 1, 2010),
    ('WALTER DIAS', 9, 2009),
    ('BRENO VENTUROSO', 1, 2009),
    ('ADELINA TEIXEIRA', 5, 2009),
    ('ELIO SILVA', 7, 2010),
    ('DENIS FONSECA', 6, 2010)
]
colNames = ['nome', 'mes', 'ano']
df = spark.createDataFrame(data, colNames)
df.show(truncate=False)

+---------------------+---+----+
|nome                 |mes|ano |
+---------------------+---+----+
|CARMINA RABELO       |4  |2010|
|HERONDINA PEREIRA    |6  |2009|
|IRANI DOS SANTOS     |12 |2010|
|JOAO BOSCO DA FONSECA|3  |2009|
|CARLITO SOUZA        |1  |2010|
|WALTER DIAS          |9  |2009|
|BRENO VENTUROSO      |1  |2009|
|ADELINA TEIXEIRA     |5  |2009|
|ELIO SILVA           |7  |2010|
|DENIS FONSECA        |6  |2010|
+---------------------+---+----+



Vendos os alunos mais novos

In [44]:
df\
    .select('*')\
    .orderBy(['ano', 'mes'], ascending=[False, False])\
    .show(truncate=False)

+---------------------+---+----+
|nome                 |mes|ano |
+---------------------+---+----+
|IRANI DOS SANTOS     |12 |2010|
|ELIO SILVA           |7  |2010|
|DENIS FONSECA        |6  |2010|
|CARMINA RABELO       |4  |2010|
|CARLITO SOUZA        |1  |2010|
|WALTER DIAS          |9  |2009|
|HERONDINA PEREIRA    |6  |2009|
|ADELINA TEIXEIRA     |5  |2009|
|JOAO BOSCO DA FONSECA|3  |2009|
|BRENO VENTUROSO      |1  |2009|
+---------------------+---+----+



#Filtrando os dados

.where() ou .filter()

São iguais

In [45]:
empresas\
.where("capital_social_da_empresa==50")\
.show(5, False)

+-----------+-----------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+
|cnpj_basico|razao_social_nome_empresarial|natureza_juridica|qualificacao_do_responsavel|capital_social_da_empresa|porte_da_empresa|ente_federativo_responsavel|
+-----------+-----------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+
+-----------+-----------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+



### Usando vários filter() em sequência

usando

.startwith(valor)

e

.endswith(valor)


para procurar strings que comecem e terminem com o valor indicado.

In [46]:
socios\
.select("nome_do_socio_ou_razao_social")\
.filter(socios.nome_do_socio_ou_razao_social.startswith("RODRIGO"))\
.filter(socios.nome_do_socio_ou_razao_social.endswith("DIAS"))\
.limit(10)\
.toPandas()

,nome_do_socio_ou_razao_social
0,RODRIGO BENASSI DIAS
1,RODRIGO RUDIBERTO DIAS
2,RODRIGO AURELIANO DIAS
3,RODRIGO SIMOES LEMOS DIAS
4,RODRIGO GEORGE DIAS
5,RODRIGO AUGUSTO FELICIO DIAS
6,RODRIGO FERNANDES DIAS
7,RODRIGO GARRIDO DIAS
8,RODRIGO OLIVEIRA DIAS
9,RODRIGO GONCALVES DIAS


In [47]:
socios\
.select("nome_do_socio_ou_razao_social")\
.filter(socios.nome_do_socio_ou_razao_social.startswith("RENAN"))\
.filter(socios.nome_do_socio_ou_razao_social.endswith("MENDES"))\
.limit(10)\
.toPandas()

,nome_do_socio_ou_razao_social
0,RENAN SIQUEIRA MENDES
1,RENAN LUIZ VILELA MENDES
2,RENAN VIEIRA MENDES
3,RENAN RAFAEL PEREIRA MENDES
4,RENAN TIBIRICA MENDES
5,RENAN LUIZ VILELA MENDES
6,RENAN PEREIRA MENDES
7,RENAN PEREIRA MENDES
8,RENAN GOMES MENDES
9,RENAN CORDEIRO MENDES


Lição

In [48]:
data = [
    ('CARMINA RABELO', 4, 2010),
    ('HERONDINA PEREIRA', 6, 2009),
    ('IRANI DOS SANTOS', 12, 2010),
    ('JOAO BOSCO DA FONSECA', 3, 2009),
    ('CARLITO SOUZA', 1, 2010),
    ('WALTER DIAS', 9, 2009),
    ('BRENO VENTUROSO', 1, 2009),
    ('ADELINA TEIXEIRA', 5, 2009),
    ('ELIO SILVA', 7, 2010),
    ('DENIS FONSECA', 6, 2010)
]
colNames = ['nome', 'mes', 'ano']
df = spark.createDataFrame(data, colNames)
df.show(truncate=False)

+---------------------+---+----+
|nome                 |mes|ano |
+---------------------+---+----+
|CARMINA RABELO       |4  |2010|
|HERONDINA PEREIRA    |6  |2009|
|IRANI DOS SANTOS     |12 |2010|
|JOAO BOSCO DA FONSECA|3  |2009|
|CARLITO SOUZA        |1  |2010|
|WALTER DIAS          |9  |2009|
|BRENO VENTUROSO      |1  |2009|
|ADELINA TEIXEIRA     |5  |2009|
|ELIO SILVA           |7  |2010|
|DENIS FONSECA        |6  |2010|
+---------------------+---+----+



uma query no formato string não existe diferença entre os operadores = e ==. Ambos podem ser utilizados para comparações neste tipo de query.

#O comando LIKE

Comando para procurar strings/sequência de caracteres

In [51]:
df = spark.createDataFrame([('RESTAURANTE DO RUI',), ('Juca restaurantes ltda',), ('Joca lanches',)], ['data'])
df.toPandas()

,data
0,RESTAURANTE DO RUI
1,Juca restaurantes ltda
2,Joca lanches


In [52]:
df\
.where(f.upper(df.data).like('%RESTAURANTE%'))\
.show(truncate=False)

+----------------------+
|data                  |
+----------------------+
|RESTAURANTE DO RUI    |
|Juca restaurantes ltda|
+----------------------+



Usando a função com like para ver todos os BARes

In [58]:
empresas\
.select('razao_social_nome_empresarial', 'natureza_juridica', 'porte_da_empresa', 'capital_social_da_empresa')\
.filter(f.upper(empresas['razao_social_nome_empresarial']).like('%BAR%'))\
.limit(5).toPandas()

,razao_social_nome_empresarial,natureza_juridica,porte_da_empresa,capital_social_da_empresa
0,CONTATOS BAR E LANCHONETE LTDA,2062,5,"0,00"
1,BAR E MERCEARIA KIT LTDA,2062,5,"0,00"
2,TRANSPORTADORA BARAO LTDA,2062,5,"0,00"
3,ANTONIO DA FONSECA GONCALVES BAR,2135,5,"0,00"
4,BARCLAYS BRASIL LTDA,2062,5,"0,00"


Lição

In [59]:
data = [
    ('CARMINA RABELO', 4, 2010),
    ('HERONDINA PEREIRA', 6, 2009),
    ('IRANI DOS SANTOS', 12, 2010),
    ('JOAO BOSCO DA FONSECA', 3, 2009),
    ('CARLITO SOUZA', 1, 2010),
    ('WALTER DIAS', 9, 2009),
    ('BRENO VENTUROSO', 1, 2009),
    ('ADELINA TEIXEIRA', 5, 2009),
    ('ELIO SILVA', 7, 2010),
    ('DENIS FONSECA', 6, 2010)
]
colNames = ['nome', 'mes', 'ano']
df = spark.createDataFrame(data, colNames)
df.show(truncate=False)

+---------------------+---+----+
|nome                 |mes|ano |
+---------------------+---+----+
|CARMINA RABELO       |4  |2010|
|HERONDINA PEREIRA    |6  |2009|
|IRANI DOS SANTOS     |12 |2010|
|JOAO BOSCO DA FONSECA|3  |2009|
|CARLITO SOUZA        |1  |2010|
|WALTER DIAS          |9  |2009|
|BRENO VENTUROSO      |1  |2009|
|ADELINA TEIXEIRA     |5  |2009|
|ELIO SILVA           |7  |2010|
|DENIS FONSECA        |6  |2010|
+---------------------+---+----+



In [60]:
df\
    .filter(df.nome.like('C%'))\
    .show(truncate=False)

+--------------+---+----+
|nome          |mes|ano |
+--------------+---+----+
|CARMINA RABELO|4  |2010|
|CARLITO SOUZA |1  |2010|
+--------------+---+----+



# Sumarizando os dados

Podemos usar

.groupby()

.count()

.orderby()



In [66]:
socios\
.select(f.year(socios.data_de_entrada_sociedade).alias('ano_de_entrada'))\
.filter('ano_de_entrada >= 2010')\
.groupBy('ano_de_entrada')\
.count()\
.orderBy('ano_de_entrada', ascending=False)\
.show()

+--------------+------+
|ano_de_entrada| count|
+--------------+------+
|          2021| 56316|
|          2020|125927|
|          2019|118248|
|          2018| 99935|
|          2017| 90221|
|          2016| 81587|
|          2015| 80906|
|          2014| 80590|
|          2013| 83919|
|          2012| 80101|
|          2011| 83906|
|          2010| 79337|
+--------------+------+



vendo capital e quantidade de empresas agrupadas pelo porte

In [68]:
empresas\
.select('cnpj_basico','porte_da_empresa', 'capital_social_da_empresa')\
.groupBy('porte_da_empresa')\
.agg(
    f.round(f.avg('capital_social_da_empresa'),2).alias('capital_medio'),
    f.count('cnpj_basico').alias('frequencia')
)\
.show()

+----------------+-------------+----------+
|porte_da_empresa|capital_medio|frequencia|
+----------------+-------------+----------+
|            NULL|         NULL|      5985|
|               1|         NULL|   3129043|
|               3|         NULL|    115151|
|               5|         NULL|   1335500|
+----------------+-------------+----------+



Podemos usar


.summary()


para ver informações estatíticas dos dados

In [69]:
empresas\
.select('capital_social_da_empresa')\
.summary()\
.show()

+-------+-------------------------+
|summary|capital_social_da_empresa|
+-------+-------------------------+
|  count|                  4585679|
|   mean|                     NULL|
| stddev|                     NULL|
|    min|                     0,00|
|    25%|                     NULL|
|    50%|                     NULL|
|    75%|                     NULL|
|    max|             999999999,99|
+-------+-------------------------+



podemos ver dados especificos como mean e max

In [73]:
empresas\
.select('capital_social_da_empresa')\
.summary('mean', 'max')\
.show()

+-------+-------------------------+
|summary|capital_social_da_empresa|
+-------+-------------------------+
|   mean|                     NULL|
|    max|             999999999,99|
+-------+-------------------------+



Lição

In [74]:
data = [
    ('CARLOS', 'MATEMÁTICA', 7),
    ('IVO', 'MATEMÁTICA', 9),
    ('MÁRCIA', 'MATEMÁTICA', 8),
    ('LEILA', 'MATEMÁTICA', 9),
    ('BRENO', 'MATEMÁTICA', 7),
    ('LETÍCIA', 'MATEMÁTICA', 8),
    ('CARLOS', 'FÍSICA', 2),
    ('IVO', 'FÍSICA', 8),
    ('MÁRCIA', 'FÍSICA', 10),
    ('LEILA', 'FÍSICA', 9),
    ('BRENO', 'FÍSICA', 1),
    ('LETÍCIA', 'FÍSICA', 6),
    ('CARLOS', 'QUÍMICA', 10),
    ('IVO', 'QUÍMICA', 8),
    ('MÁRCIA', 'QUÍMICA', 1),
    ('LEILA', 'QUÍMICA', 10),
    ('BRENO', 'QUÍMICA', 7),
    ('LETÍCIA', 'QUÍMICA', 9)
]
colNames = ['nome', 'materia', 'nota']
df = spark.createDataFrame(data, colNames)
df.show()

+-------+----------+----+
|   nome|   materia|nota|
+-------+----------+----+
| CARLOS|MATEMÁTICA|   7|
|    IVO|MATEMÁTICA|   9|
| MÁRCIA|MATEMÁTICA|   8|
|  LEILA|MATEMÁTICA|   9|
|  BRENO|MATEMÁTICA|   7|
|LETÍCIA|MATEMÁTICA|   8|
| CARLOS|    FÍSICA|   2|
|    IVO|    FÍSICA|   8|
| MÁRCIA|    FÍSICA|  10|
|  LEILA|    FÍSICA|   9|
|  BRENO|    FÍSICA|   1|
|LETÍCIA|    FÍSICA|   6|
| CARLOS|   QUÍMICA|  10|
|    IVO|   QUÍMICA|   8|
| MÁRCIA|   QUÍMICA|   1|
|  LEILA|   QUÍMICA|  10|
|  BRENO|   QUÍMICA|   7|
|LETÍCIA|   QUÍMICA|   9|
+-------+----------+----+



Usando

.when() e .otherwise()


para criar uma coluna lógica

In [75]:
df = df.withColumn('status', f.when(df.nota >= 7, "APROVADO").otherwise("REPROVADO"))
df.show()

+-------+----------+----+---------+
|   nome|   materia|nota|   status|
+-------+----------+----+---------+
| CARLOS|MATEMÁTICA|   7| APROVADO|
|    IVO|MATEMÁTICA|   9| APROVADO|
| MÁRCIA|MATEMÁTICA|   8| APROVADO|
|  LEILA|MATEMÁTICA|   9| APROVADO|
|  BRENO|MATEMÁTICA|   7| APROVADO|
|LETÍCIA|MATEMÁTICA|   8| APROVADO|
| CARLOS|    FÍSICA|   2|REPROVADO|
|    IVO|    FÍSICA|   8| APROVADO|
| MÁRCIA|    FÍSICA|  10| APROVADO|
|  LEILA|    FÍSICA|   9| APROVADO|
|  BRENO|    FÍSICA|   1|REPROVADO|
|LETÍCIA|    FÍSICA|   6|REPROVADO|
| CARLOS|   QUÍMICA|  10| APROVADO|
|    IVO|   QUÍMICA|   8| APROVADO|
| MÁRCIA|   QUÍMICA|   1|REPROVADO|
|  LEILA|   QUÍMICA|  10| APROVADO|
|  BRENO|   QUÍMICA|   7| APROVADO|
|LETÍCIA|   QUÍMICA|   9| APROVADO|
+-------+----------+----+---------+



# Juntando DataFrames

In [76]:
produtos = spark.createDataFrame(
    [
        ('1', 'Bebidas', 'Agua mineral'),
        ('2', 'Limpeza', 'Sabão em pó'),
        ('3', 'Frios', 'Queijo'),
        ('4', 'Pet', 'Ração'),
    ],
    ['id', 'categoria', 'produto']
)
produtos.show()


+---+---------+------------+
| id|categoria|     produto|
+---+---------+------------+
|  1|  Bebidas|Agua mineral|
|  2|  Limpeza| Sabão em pó|
|  3|    Frios|      Queijo|
|  4|      Pet|       Ração|
+---+---------+------------+



In [78]:
impostos = spark.createDataFrame(
    [
        ('Bebidas', 0.15),
        ('Limpeza', 0.05),
        ('Frios', 0.065),
        ('Carnes', 0.08),
    ],
    ['categoria', 'imposto']
)
impostos.show()


+---------+-------+
|categoria|imposto|
+---------+-------+
|  Bebidas|   0.15|
|  Limpeza|   0.05|
|    Frios|  0.065|
|   Carnes|   0.08|
+---------+-------+



##Fazendo joins

In [81]:
produtos.join(impostos, 'categoria', how = 'inner')\
.sort('id')\
.show()

+---------+---+------------+-------+
|categoria| id|     produto|imposto|
+---------+---+------------+-------+
|  Bebidas|  1|Agua mineral|   0.15|
|  Limpeza|  2| Sabão em pó|   0.05|
|    Frios|  3|      Queijo|  0.065|
+---------+---+------------+-------+



In [82]:
produtos.join(impostos, 'categoria', how = 'left')\
.sort('id')\
.show()

+---------+---+------------+-------+
|categoria| id|     produto|imposto|
+---------+---+------------+-------+
|  Bebidas|  1|Agua mineral|   0.15|
|  Limpeza|  2| Sabão em pó|   0.05|
|    Frios|  3|      Queijo|  0.065|
|      Pet|  4|       Ração|   NULL|
+---------+---+------------+-------+



In [83]:
produtos.join(impostos, 'categoria', how = 'right')\
.sort('id')\
.show()

+---------+----+------------+-------+
|categoria|  id|     produto|imposto|
+---------+----+------------+-------+
|   Carnes|NULL|        NULL|   0.08|
|  Bebidas|   1|Agua mineral|   0.15|
|  Limpeza|   2| Sabão em pó|   0.05|
|    Frios|   3|      Queijo|  0.065|
+---------+----+------------+-------+



In [84]:
produtos.join(impostos, 'categoria', how = 'outer')\
.sort('id')\
.show()

+---------+----+------------+-------+
|categoria|  id|     produto|imposto|
+---------+----+------------+-------+
|   Carnes|NULL|        NULL|   0.08|
|  Bebidas|   1|Agua mineral|   0.15|
|  Limpeza|   2| Sabão em pó|   0.05|
|    Frios|   3|      Queijo|  0.065|
|      Pet|   4|       Ração|   NULL|
+---------+----+------------+-------+



cnpj_basico é a chave estrangeira/ em comum entre as tabelas

In [85]:
socios.printSchema()

root
 |-- cnpj_basico: integer (nullable = true)
 |-- identificador_de_socio: integer (nullable = true)
 |-- nome_do_socio_ou_razao_social: string (nullable = true)
 |-- cnpj_ou_cpf_do_socio: string (nullable = true)
 |-- qualificacao_do_socio: integer (nullable = true)
 |-- data_de_entrada_sociedade: date (nullable = true)
 |-- pais: integer (nullable = true)
 |-- representante_legal: string (nullable = true)
 |-- nome_do_representante: string (nullable = true)
 |-- qualificacao_do_representante_legal: integer (nullable = true)
 |-- faixa_etaria: integer (nullable = true)



In [86]:
empresas.printSchema()

root
 |-- cnpj_basico: integer (nullable = true)
 |-- razao_social_nome_empresarial: string (nullable = true)
 |-- natureza_juridica: integer (nullable = true)
 |-- qualificacao_do_responsavel: integer (nullable = true)
 |-- capital_social_da_empresa: string (nullable = true)
 |-- porte_da_empresa: integer (nullable = true)
 |-- ente_federativo_responsavel: string (nullable = true)



In [88]:
estabelecimentos.printSchema()

root
 |-- cnpj_basico: integer (nullable = true)
 |-- cnpj_ordem: integer (nullable = true)
 |-- cnpj_dv: integer (nullable = true)
 |-- identificador_matriz_filial: integer (nullable = true)
 |-- nome_fantasia: string (nullable = true)
 |-- situacao_cadastral: integer (nullable = true)
 |-- data_situacao_cadastral: date (nullable = true)
 |-- motivo_situacao_cadastral: integer (nullable = true)
 |-- nome_da_cidade_no_exterior: string (nullable = true)
 |-- pais: integer (nullable = true)
 |-- data_de_inicio_atividade: date (nullable = true)
 |-- cnae_fiscal_principal: integer (nullable = true)
 |-- cnae_fiscal_secundaria: string (nullable = true)
 |-- tipo_de_logradouro: string (nullable = true)
 |-- logradouro: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- complemento: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- cep: integer (nullable = true)
 |-- uf: string (nullable = true)
 |-- municipio: integer (nullable = true)
 |-- ddd_1: string (n

Criando um df com o join de empresas e estabelecimentos

In [90]:
empresas_join = estabelecimentos.join(empresas, 'cnpj_basico', how = 'inner')

In [91]:
empresas_join.printSchema()

root
 |-- cnpj_basico: integer (nullable = true)
 |-- cnpj_ordem: integer (nullable = true)
 |-- cnpj_dv: integer (nullable = true)
 |-- identificador_matriz_filial: integer (nullable = true)
 |-- nome_fantasia: string (nullable = true)
 |-- situacao_cadastral: integer (nullable = true)
 |-- data_situacao_cadastral: date (nullable = true)
 |-- motivo_situacao_cadastral: integer (nullable = true)
 |-- nome_da_cidade_no_exterior: string (nullable = true)
 |-- pais: integer (nullable = true)
 |-- data_de_inicio_atividade: date (nullable = true)
 |-- cnae_fiscal_principal: integer (nullable = true)
 |-- cnae_fiscal_secundaria: string (nullable = true)
 |-- tipo_de_logradouro: string (nullable = true)
 |-- logradouro: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- complemento: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- cep: integer (nullable = true)
 |-- uf: string (nullable = true)
 |-- municipio: integer (nullable = true)
 |-- ddd_1: string (n

Vendo a frequencia de inicio de empresas, a partir de 2010

In [92]:
freq = empresas_join\
.select('cnpj_basico', f.year('data_de_inicio_atividade').alias('ano_de_inicio'))\
.where('ano_de_inicio >= 2010')\
.groupBy('ano_de_inicio')\
.agg(f.count('cnpj_basico').alias('frequencia'))\
.orderBy('ano_de_inicio', ascending=False)

In [93]:
freq.toPandas()

,ano_de_inicio,frequencia
0,2021,153275
1,2020,400654
2,2019,325922
3,2018,275435
4,2017,237292
5,2016,265417
6,2015,212523
7,2014,202276
8,2013,198424
9,2012,232480


com


.lit()


podemos adicionar uma linha de total em baixo

In [95]:
freq.union(
    freq.select(
        f.lit('Total').alias('ano_de_inicio'),
        f.sum('frequencia').alias('frequencia')
)
    ).show()

+-------------+----------+
|ano_de_inicio|frequencia|
+-------------+----------+
|         2021|    153275|
|         2020|    400654|
|         2019|    325922|
|         2018|    275435|
|         2017|    237292|
|         2016|    265417|
|         2015|    212523|
|         2014|    202276|
|         2013|    198424|
|         2012|    232480|
|         2011|    172677|
|         2010|    154159|
|        Total|   2830534|
+-------------+----------+



Lição

In [96]:
idades = spark.createDataFrame(
    [
        ('CARLOS', 15),
        ('IVO', 14),
        ('MÁRCIA', 16),
        ('LEILA', 17),
        ('LETÍCIA', 14)
    ],
    ['nomes', 'idades']
)

notas = spark.createDataFrame(
    [
        ('CARLOS', 10),
        ('MÁRCIA', 1),
        ('LEILA', 10),
        ('BRENO', 7),
        ('LETÍCIA', 9)
    ],
    ['nomes', 'notas']
)

In [97]:
idades.join(notas, 'nomes', how='outer')\
    .sort('nomes')\
    .show()

+-------+------+-----+
|  nomes|idades|notas|
+-------+------+-----+
|  BRENO|  NULL|    7|
| CARLOS|    15|   10|
|    IVO|    14| NULL|
|  LEILA|    17|   10|
|LETÍCIA|    14|    9|
| MÁRCIA|    16|    1|
+-------+------+-----+



#SparkSQL

Criando views e

.sql()


podemos usar queries para criar df/tabelas

In [98]:
empresas.createOrReplaceTempView('empresasView') # Criando uma view SQL

In [99]:
spark.sql('SELECT * FROM empresasView').show(5)

+-----------+-----------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+
|cnpj_basico|razao_social_nome_empresarial|natureza_juridica|qualificacao_do_responsavel|capital_social_da_empresa|porte_da_empresa|ente_federativo_responsavel|
+-----------+-----------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+
|        306|         FRANCAMAR REFRIGE...|             2240|                         49|                     0,00|               1|                       NULL|
|       1355|         BRASILEIRO & OLIV...|             2062|                         49|                     0,00|               5|                       NULL|
|       4820|         REGISTRO DE IMOVE...|             3034|                         32|                     0,00|               5|                       NULL|
|       5347|         ROSELY APARE

In [105]:
spark\
.sql('SELECT * FROM empresasView WHERE capital_social_da_empresa = 4000')\
.show(5)

+-----------+-----------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+
|cnpj_basico|razao_social_nome_empresarial|natureza_juridica|qualificacao_do_responsavel|capital_social_da_empresa|porte_da_empresa|ente_federativo_responsavel|
+-----------+-----------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+
+-----------+-----------------------------+-----------------+---------------------------+-------------------------+----------------+---------------------------+



In [107]:
spark\
.sql('SELECT porte_da_empresa, MEAN(capital_social_da_empresa) as MEDIA FROM empresasView GROUP BY porte_da_empresa')\
.show(5)


+----------------+-----+
|porte_da_empresa|MEDIA|
+----------------+-----+
|            NULL| NULL|
|               1| NULL|
|               3| NULL|
|               5| NULL|
+----------------+-----+



In [108]:
empresas_join.createOrReplaceTempView('empresasJoinView') # Criando uma view de Join

In [113]:
freq = spark.sql("""
    SELECT YEAR(data_de_inicio_atividade) AS ano_de_inicio, COUNT(cnpj_basico) AS Freq
    FROM empresasJoinView
    WHERE YEAR(data_de_inicio_atividade) >= 2010
    GROUP BY ano_de_inicio
    ORDER BY ano_de_inicio
""")

freq\
.show()

+-------------+------+
|ano_de_inicio|  Freq|
+-------------+------+
|         2010|154159|
|         2011|172677|
|         2012|232480|
|         2013|198424|
|         2014|202276|
|         2015|212523|
|         2016|265417|
|         2017|237292|
|         2018|275435|
|         2019|325922|
|         2020|400654|
|         2021|153275|
+-------------+------+



#Armazenamento

##Arquivos CSV

In [114]:
empresas.write.csv(
    path = '/content/drive/MyDrive/Curso_Spark/empresas/CSV',
    mode ='overwrite',
    header =True,
    sep =';',
    encoding ='UTF-8'
)

##Arquivos Parquet

In [115]:
empresas.write.parquet(
    path = '/content/drive/MyDrive/Curso_Spark/empresas/Parquet',
    mode ='overwrite'
)

lendo um arquivo parquet

In [116]:
empresas_parquet = spark.read.parquet(
 '/content/drive/MyDrive/Curso_Spark/empresas/Parquet'
)

##Arquivos ORC

In [ ]:
empresas.write.orc('/content/drive/MyDrive/data/empresas/orc/')

##Particionamento dos dados

Os particionamentos são realizados automaticamente por default

Criando uma partição única com coalesce

existe também o repartition() mas é mais pesado, e ele permite aumentar ou diminuir a repartição

In [ ]:
empresas.coalesce(1).write.csv(
    path = '/content/drive/MyDrive/Curso_Spark/empresas/CSV_unico',
    mode ='overwrite',
    sep=';'.
    header = True
)

Podemos criar partições por coluna, usando


partitionby

In [ ]:
empresas.write.parquet(
    path = '/content/drive/MyDrive/Curso_Spark/empresas/Parquet-partitionby',
    mode ='overwrite',
    partitionby = 'porte_da_empresa'
)

#Fechando uma sessão Spark

In [117]:
spark.stop()